# Cat-Dog Image Classifier

### Importing Libraries

In [24]:
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory  # pyright: ignore
from tensorflow.keras import layers, Sequential # pyright: ignore
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint # pyright: ignore

### Loading Images

In [25]:
train = image_dataset_from_directory(
    '../Data/catdog/training_set',
    validation_split = 0.2,
    subset = 'training',
    seed = 123,
    image_size = (180, 180),
    batch_size = 32
)

validation = image_dataset_from_directory(
    '../Data/catdog/training_set',
    validation_split = 0.2,
    subset = 'validation',
    seed = 123,
    image_size = (180, 180),
    batch_size = 32
)


test = image_dataset_from_directory(
    '../Data/catdog/test_set',
    seed = 123,
    image_size = (180, 180),
    batch_size = 32
)

Found 8000 files belonging to 2 classes.
Using 6400 files for training.
Found 8000 files belonging to 2 classes.
Using 1600 files for validation.
Found 2000 files belonging to 2 classes.


### Scaling and augmenting 

In [26]:
augmenting = Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])
rescale = layers.Rescaling(1./255)

### Building The Model

In [27]:
model = Sequential([
    augmenting,
    rescale,
    layers.Conv2D(32,3, activation= 'relu', input_shape = (180,180,3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64,3, activation= 'relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128,3, activation= 'relu'),
    layers.MaxPooling2D(),
    layers.Dense(128, activation= 'relu'),
    layers.Dropout(0.4),
    layers.Dense(1, activation= 'sigmoid')
])

d:\Work\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### Compiling the model

In [28]:
model.compile(
    optimizer = 'adam',
    loss = 'binary_crossentropy',
    metrics = ['accuracy']
)

### Training the Model

In [29]:
early_stop = EarlyStopping(monitor= 'val_loss', patience= 5, restore_best_weights= True)
checkpoint = ModelCheckpoint('best_model.keras', monitor= 'val_accuracy', save_best_only= True)

history = model.fit(
    train,
    validation_data  = validation,
    epochs = 50,
    callbacks = [early_stop, checkpoint]
)

Epoch 1/50


d:\Work\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


ValueError: Arguments `target` and `output` must have the same rank (ndim). Received: target.shape=(None,), output.shape=(None, 20, 20, 1)